# Customer Review Sentiment Analysis – Model Training

This notebook contains the end-to-end model training pipeline for the 
Customer Review Insight System deployed using Streamlit.

Dataset: Amazon Fine Food Reviews
Task: Binary Sentiment Classification (Positive vs Negative)

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df=pd.read_csv(r"C:\Users\mohit\OneDrive\Desktop\Data_for_model\Project\Amazon_food\Reviews.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.duplicated().sum()

## Labeling output

In [ ]:
def label(score):
    if score<=2:
        return "Negative"
    elif score==3:
        return "Neutral"
    else:
        return "Positive"
    

In [ ]:
df['Sentiment']=df['Score'].apply(label)

In [ ]:
df.head()

In [ ]:
df=df[df['Sentiment']!='Neutral']  # Considering only positive & negative reviews 

In [ ]:
df=df[['Text','Sentiment']]
df

In [ ]:
df['Sentiment'].value_counts()

In [ ]:
import sys
!{sys.executable} -m pip install nltk

## Text Processing

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
stop_words=set(stopwords.words('english'))
negations = {"not", "no", "nor", "never"}
stop_words = stop_words - negations
lemmitizer=WordNetLemmatizer()

def clean_text(text):
    text=text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text=text.split()
    text = [lemmitizer.lemmatize(word) for word in text if word not in stop_words]
    text=" ".join(text)
    return text

In [ ]:
df['Text']=df['Text'].apply(clean_text)

In [ ]:
df

In [ ]:
df['Sentiment']=df['Sentiment'].map({'Positive':1, 'Negative':0})

## Train Test split

In [ ]:
from sklearn.model_selection import train_test_split
X=df['Text']
y=df['Sentiment']

X_train, X_test,y_train, y_test= train_test_split(X,y,test_size=0.2,random_state=42)

## TF-IDF with Bigrams (Important)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfid=TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfid=tfid.fit_transform(X_train)
X_test_tfid=tfid.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight='balanced')
}

for name, model in models.items():
    
    print("====================================")
    print("Model:", name)
    
    model.fit(X_train_tfid, y_train)
    y_pred = model.predict(X_test_tfid)
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

# Choosing logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model =LogisticRegression(class_weight='balanced')

model.fit(X_train_tfid,y_train)

In [ ]:
y_pred=model.predict(X_test_tfid)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import pickle
pickle.dump(model,open('model.pkl','wb'))
pickle.dump(tfid,open('vectorizer.pkl','wb'))